# Jiema Sunflower-Gemma4-E2B Quantization Spike

Goal: test whether `Sunbird/Sunflower-Gemma4-E2B` can run in 4-bit on Colab, and whether the output is usable for Jiema's Wolof-first assistant.

This notebook is a feasibility spike, not the final Android packaging path. It checks:

- Can the model load in 4-bit?
- What are latency and memory like?
- Does it return useful Wolof answers for a few Jiema prompts?
- Does it obey JSON output enough for the app backend?


## Runtime

Use a GPU runtime. Recommended: L4, A100, or T4 if nothing else is available.

In Colab: `Runtime -> Change runtime type -> GPU`.


In [ ]:
!nvidia-smi
!df -h


## Install Dependencies

Use recent Transformers because Sunflower-Gemma4-E2B needs `AutoModelForMultimodalLM`.


In [ ]:
!pip -q install -U 'transformers>=5.14.0' accelerate bitsandbytes safetensors huggingface_hub


In [ ]:
import json, time, os, gc
from pathlib import Path
import torch
import transformers
print('torch', torch.__version__)
print('transformers', transformers.__version__)
print('cuda available', torch.cuda.is_available())


## Hugging Face Login

You must have accepted access to `Sunbird/Sunflower-Gemma4-E2B` on Hugging Face first.

Paste a Hugging Face token when prompted.


In [ ]:
from huggingface_hub import notebook_login
notebook_login()


## Load Sunflower-Gemma4-E2B In 4-bit

This uses bitsandbytes runtime quantization. It is useful for feasibility and latency testing, but it is not yet an Android artifact.


In [ ]:
from transformers import AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig

MODEL_ID = 'Sunbird/Sunflower-Gemma4-E2B'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    low_cpu_mem_usage=True,
).eval()
load_seconds = time.time() - t0
print('load_seconds', round(load_seconds, 2))
print('device map', getattr(model, 'hf_device_map', None))


In [ ]:
def gpu_memory():
    if not torch.cuda.is_available():
        return {}
    return {
        'allocated_gb': round(torch.cuda.memory_allocated() / 1024**3, 3),
        'reserved_gb': round(torch.cuda.memory_reserved() / 1024**3, 3),
        'max_allocated_gb': round(torch.cuda.max_memory_allocated() / 1024**3, 3),
    }

gpu_memory()


## Jiema Prompt

This is intentionally short. The Mac test showed that long JSON-heavy prompting made the model unstable.


In [ ]:
SYSTEM_PROMPT = (
    'You are Jiema, a helpful assistant for Senegalese users. '
    'Wolof is your priority language, but keep your ability to understand and answer other African languages. '
    'When the user writes in Wolof, French, English, or mixed Wolof-French, answer in simple spoken Wolof. '
    'Answer the question directly. Do not repeat or translate the question.'
)

def user_prompt(text):
    return text


In [ ]:
def normalize_text(text):
    return text.replace('<end_of_turn>', '').replace('<bos>', '').replace('<eos>', '').strip()

def generate_jiema(text, max_new_tokens=120):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user_prompt(text)},
    ]
    try:
        prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(
        text=[prompt],
        return_tensors='pt',
        text_kwargs={'padding': False, 'truncation': True, 'add_special_tokens': False},
    ).to(model.device)
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
        )
    latency = time.time() - t0
    new_tokens = outputs[0][inputs['input_ids'].shape[-1]:]
    raw = processor.decode(new_tokens, skip_special_tokens=False)
    try:
        content = processor.parse_response(raw)['content']
    except Exception:
        content = raw
    content = normalize_text(content)
    normalized_input = ' '.join(text.lower().split()).rstrip('?.!')
    normalized_output = ' '.join(content.lower().split()).rstrip('?.!')
    return {
        'latency_s': round(latency, 3),
        'tokens_generated': int(new_tokens.shape[-1]),
        'gpu_memory': gpu_memory(),
        'answer': content,
        'echoed_input': normalized_output == normalized_input,
    }


## Five-Prompt Smoke Test


In [ ]:
TEST_PROMPTS = [
    {'id': 'TRAN_001', 'category': 'transport', 'prompt': 'Ana bus bi dem Sandaga?'},
    {'id': 'AGRI_001', 'category': 'agriculture', 'prompt': 'Xob yi ci sama gerte dañuy weex. Lan laa wara def?'},
    {'id': 'HEALTH_001', 'category': 'health', 'prompt': 'Sama doom dafa am tàngoor bu metti. Lan laa wara def?'},
    {'id': 'SMS_005', 'category': 'admin', 'prompt': 'Un message me demande mon code secret Orange Money. Explique en wolof quoi faire.'},
    {'id': 'MIXED_001', 'category': 'general', 'prompt': 'Dama am problème avec formulaire bi, je ne comprends pas adresse permanente.'},
]

results = []
for item in TEST_PROMPTS:
    print('running', item['id'])
    out = generate_jiema(item['prompt'], max_new_tokens=120)
    row = {**item, **out}
    results.append(row)
    print(json.dumps(row, ensure_ascii=False, indent=2)[:2000])

summary = {
    'model': MODEL_ID,
    'quantization': 'bitsandbytes_nf4_4bit',
    'load_seconds': round(load_seconds, 2),
    'avg_latency_s': round(sum(r['latency_s'] for r in results) / len(results), 3),
    'max_latency_s': max(r['latency_s'] for r in results),
    'memory_after_eval': gpu_memory(),
}
report = {'summary': summary, 'results': results}
Path('jiema_sunflower_4bit_smoke_results.json').write_text(json.dumps(report, ensure_ascii=False, indent=2))
summary


## Optional: Save To Google Drive


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp jiema_sunflower_4bit_smoke_results.json /content/drive/MyDrive/


## Interpretation

Good sign:

- Model loads in 4-bit without OOM.
- Average latency is low enough for interaction.
- Answers address the user's question instead of repeating it.
- Wolof answers are useful enough for native-speaker review.

Bad sign:

- OOM during load.
- Outputs are empty, incomplete, or copied prompts.
- Latency is still too slow.
- Health answers make unsafe claims.

If the smoke test passes, next step is to export a real artifact: GGUF, MLX, LiteRT, or another mobile-compatible format. bitsandbytes 4-bit is only a feasibility check.
